# 4 Macro Simulation Scenario

This analysis notebook loads a baseline basin risk table and a scenario basin
risk table, builds paired curve dictionaries, and runs the macro flood shock
simulation using the new baseline/scenario workflow.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import numpy as np
import pandas as pd

from sovereign.flood import BasinLossCurve, build_basin_curves
from sovereign.macroeconomic import prepare_DIGNAD, run_DIGNAD, run_flood_sim_for_macro


In [5]:
# USER CONFIG
model = "wri"
n_years = 10000
scenario_mode = "combined"
scenario_name = "pubinf_urban_mask_50pct_nbs_shift_150pct"

# Flood-to-macro scaling inputs
UGA_GDP = 54e9
agr_GVA = 14100000000
man_GVA = 14240000000
ser_GVA = 24710000000
TRADABLE_SHARES = {
    "Agriculture": 1.0,
    "Manufacturing": 0.7,
    "Service": 0.5,
}

# Optional DIGNAD settings
run_dignad = True
adaptation_cost = 7.74
sim_start_year = 2022
nat_disaster_year = 2027
recovery_period = 3
reconstruction_efficiency = 0
public_debt_premium = 0
gdp_avg_years = 5


In [3]:
# Paths and baseline inputs
root = Path.cwd().parent
calibration_path = root / "inputs" / "macro" / "UGA_2024_calibration_final.csv"
baseline_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"
scenario_basin_path = root / "outputs" / "flood" / "adaptation" / scenario_mode / scenario_name / "basins" / f"risk_basins_m-{model}.csv"
copula_path = root / "outputs" / "flood" / "dependence" / "copulas" / "copula_random_numbers.gzip"

baseline_risk_data = pd.read_csv(baseline_basin_path)
baseline_risk_data = baseline_risk_data.iloc[:, 1:] if str(baseline_risk_data.columns[0]).startswith("Unnamed") else baseline_risk_data
baseline_risk_data["AEP"] = 1 / baseline_risk_data["RP"]
baseline_risk_data["Pr_L_AEP"] = np.where(baseline_risk_data["Pr_L"] == 0, 0, 1 / baseline_risk_data["Pr_L"])
baseline_risk_data.reset_index(drop=True, inplace=True)

scenario_risk_data = pd.read_csv(scenario_basin_path)
scenario_risk_data = scenario_risk_data.iloc[:, 1:] if str(scenario_risk_data.columns[0]).startswith("Unnamed") else scenario_risk_data
if "AEP" not in scenario_risk_data.columns:
    scenario_risk_data["AEP"] = 1 / scenario_risk_data["RP"]
if "Pr_L_AEP" not in scenario_risk_data.columns:
    scenario_risk_data["Pr_L_AEP"] = np.where(scenario_risk_data["Pr_L"] == 0, 0, 1 / scenario_risk_data["Pr_L"])
scenario_risk_data.reset_index(drop=True, inplace=True)

copula_random_numbers = pd.read_parquet(copula_path).iloc[:n_years].copy()

baseline_risk_data.head()


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
2,2,UGA.27_1,Kitgum,1.060999e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
3,3,UGA.41_1,Moyo,1.061033e+09,2.0,0.000000,0.000000,5,Public,0.2,0.5
4,4,UGA.3_1,Arua,1.061033e+09,2.0,6160.324707,6160.324707,5,Public,0.2,0.5


In [4]:
# Build baseline and scenario curves
baseline_curves: dict[int, BasinLossCurve] = build_basin_curves(baseline_risk_data)
scenario_curves: dict[int, BasinLossCurve] = build_basin_curves(scenario_risk_data)

baseline_shocks, scenario_shocks = run_flood_sim_for_macro(
    baseline_curves,
    scenario_curves,
    n_years,
    copula_random_numbers,
    agr_GVA,
    man_GVA,
    ser_GVA,
    TRADABLE_SHARES,
    UGA_GDP,
)


100%|███████████████████████████████████████████████████████████████████████████| 10000/10000 [01:33<00:00, 107.04it/s]


In [7]:
# Compare mean macro shock inputs
comparison_df = pd.DataFrame({
    "metric": ["dY_T", "dY_N", "dK_priv", "dK_pub"],
    "baseline_mean": [baseline_shocks[c].mean() for c in ["dY_T", "dY_N", "dK_priv", "dK_pub"]],
    "scenario_mean": [scenario_shocks[c].mean() for c in ["dY_T", "dY_N", "dK_priv", "dK_pub"]],
})
comparison_df["mean_change"] = comparison_df["scenario_mean"] - comparison_df["baseline_mean"]
comparison_df["pct_change"] = np.where(
    comparison_df["baseline_mean"] != 0,
    100 * comparison_df["mean_change"] / comparison_df["baseline_mean"],
    np.nan,
)
comparison_df


,metric,baseline_mean,scenario_mean,mean_change,pct_change
0,dY_T,0.000896,0.000892,-3.350593e-06,-0.374114
1,dY_N,0.000814,0.000814,-8.468296e-08,-0.010400
2,dK_priv,0.000652,0.000648,-3.054013e-06,-0.468739
3,dK_pub,0.000385,0.000337,-4.842968e-05,-12.577240


In [11]:
# Optional: run DIGNAD on mean baseline/scenario shocks
if run_dignad:
    prepare_DIGNAD(str(calibration_path), adaptation_cost)

    baseline_gdp_impact, baseline_years = run_DIGNAD(
        sim_start_year,
        nat_disaster_year,
        recovery_period,
        baseline_shocks["dY_T"].max(),
        baseline_shocks["dY_N"].max(),
        reconstruction_efficiency,
        public_debt_premium,
        baseline_shocks["dK_pub"].max(),
        baseline_shocks["dK_priv"].max(),
        0.5,
    )

    scenario_gdp_impact, scenario_years = run_DIGNAD(
        sim_start_year,
        nat_disaster_year,
        recovery_period,
        scenario_shocks["dY_T"].max(),
        scenario_shocks["dY_N"].max(),
        reconstruction_efficiency,
        public_debt_premium,
        scenario_shocks["dK_pub"].max(),
        scenario_shocks["dK_priv"].max(),
        0.5,
    )

    dignad_df = pd.DataFrame({
        "year": baseline_years,
        "baseline_gdp_impact": baseline_gdp_impact,
        "scenario_gdp_impact": scenario_gdp_impact,
    })
    dignad_df["impact_change"] = dignad_df["scenario_gdp_impact"] - dignad_df["baseline_gdp_impact"]
    dignad_df
else:
    print("Set run_dignad = True to run the optional DIGNAD step.")


In [12]:
dignad_df

,year,baseline_gdp_impact,scenario_gdp_impact,impact_change
0,2022,0.000000,0.000000,0.000000
1,2023,0.002820,0.002797,-0.000023
2,2024,0.007309,0.007162,-0.000147
3,2025,0.012201,0.011934,-0.000267
4,2026,0.017731,0.017348,-0.000383
5,2027,-1.388620,-1.403251,-0.014631
6,2028,-1.300964,-1.297530,0.003434
7,2029,-1.007103,-1.003175,0.003928
8,2030,-0.657187,-0.652030,0.005157
9,2031,-0.486432,-0.480968,0.005465
